# 第14章：MCP 协议（Model Context Protocol）

本章介绍 Model Context Protocol（MCP），这是一个用于连接 AI 模型与外部数据源和工具的开放协议。通过模拟实现 MCP 服务器和客户端，帮助你理解其核心概念：工具注册、资源管理、资源订阅和消息传递。

**核心知识点**：
- MCP 协议架构与基本概念
- 服务器端工具和资源的定义与注册
- 客户端与服务器的交互流程
- LangChain 与 MCP 的集成方式

## 学习目标与环境准备

**学习目标**：
1. 理解 MCP 协议的核心组件和工作原理
2. 掌握 MCP 服务器的创建和工具注册
3. 学会使用 MCP 客户端调用服务器功能
4. 能够将 MCP 与 LangChain 集成使用

**环境准备**：本章通过模拟代码实现 MCP 概念，无需安装额外依赖。

## 14.1 MCP 协议核心概念

MCP 协议定义了服务器与客户端之间的标准通信方式，支持工具调用、资源管理和资源订阅等功能。下面通过模拟代码展示 MCP 的核心组件。

In [ ]:
from typing import Dict, Any, List, Callable, Optional
from dataclasses import dataclass, field
from enum import Enum, auto
import json


class MCPMessageType(Enum):
    REQUEST = auto()
    RESPONSE = auto()
    NOTIFICATION = auto()
    ERROR = auto()


@dataclass
class MCPMessage:
    type: MCPMessageType
    method: Optional[str] = None
    params: Optional[Dict[str, Any]] = None
    result: Optional[Any] = None
    error: Optional[str] = None
    id: Optional[int] = None


@dataclass
class MCPTool:
    name: str
    description: str
    input_schema: Dict[str, Any]
    handler: Callable


@dataclass
class MCPResource:
    uri: str
    name: str
    description: str
    mime_type: str
    content: Any


print("=== MCP 核心数据结构 ===")
print("MCPMessageType 定义了消息类型：REQUEST, RESPONSE, NOTIFICATION, ERROR")
print("MCPMessage 封装了完整的通信消息")
print("MCPTool 代表可调用的工具")
print("MCPResource 代表可访问的资源")

## 14.2 MCP 服务器实现

MCP 服务器负责注册工具和资源，处理客户端请求。下面实现一个简化的 MCP 服务器，支持工具调用、资源读取和资源列表。

In [ ]:
class MCPServer:
    def __init__(self, name: str):
        self.name = name
        self.tools: Dict[str, MCPTool] = {}
        self.resources: Dict[str, MCPResource] = {}
        self.subscribers: List[Callable] = []
        self.request_id: int = 0

    def register_tool(self, tool: MCPTool):
        self.tools[tool.name] = tool
        print(f"[Server] 注册工具: {tool.name}")

    def register_resource(self, resource: MCPResource):
        self.resources[resource.uri] = resource
        print(f"[Server] 注册资源: {resource.uri}")

    def handle_request(self, message: MCPMessage) -> MCPMessage:
        print(f"[Server] 收到请求: {message.method}")
        
        try:
            if message.method == "tools/list":
                return self._list_tools(message)
            elif message.method == "tools/call":
                return self._call_tool(message)
            elif message.method == "resources/list":
                return self._list_resources(message)
            elif message.method == "resources/read":
                return self._read_resource(message)
            elif message.method == "ping":
                return MCPMessage(
                    type=MCPMessageType.RESPONSE,
                    result={"status": "ok", "server": self.name},
                    id=message.id
                )
            else:
                return MCPMessage(
                    type=MCPMessageType.ERROR,
                    error=f"未知方法: {message.method}",
                    id=message.id
                )
        except Exception as e:
            return MCPMessage(
                type=MCPMessageType.ERROR,
                error=str(e),
                id=message.id
            )

    def _list_tools(self, message: MCPMessage) -> MCPMessage:
        tools_list = [
            {
                "name": t.name,
                "description": t.description,
                "inputSchema": t.input_schema
            }
            for t in self.tools.values()
        ]
        return MCPMessage(
            type=MCPMessageType.RESPONSE,
            result={"tools": tools_list},
            id=message.id
        )

    def _call_tool(self, message: MCPMessage) -> MCPMessage:
        tool_name = message.params.get("name")
        arguments = message.params.get("arguments", {})
        
        if tool_name not in self.tools:
            return MCPMessage(
                type=MCPMessageType.ERROR,
                error=f"工具不存在: {tool_name}",
                id=message.id
            )
        
        tool = self.tools[tool_name]
        result = tool.handler(**arguments)
        
        return MCPMessage(
            type=MCPMessageType.RESPONSE,
            result={"content": [{"type": "text", "text": str(result)}]},
            id=message.id
        )

    def _list_resources(self, message: MCPMessage) -> MCPMessage:
        resources_list = [
            {
                "uri": r.uri,
                "name": r.name,
                "description": r.description,
                "mimeType": r.mime_type
            }
            for r in self.resources.values()
        ]
        return MCPMessage(
            type=MCPMessageType.RESPONSE,
            result={"resources": resources_list},
            id=message.id
        )

    def _read_resource(self, message: MCPMessage) -> MCPMessage:
        uri = message.params.get("uri")
        
        if uri not in self.resources:
            return MCPMessage(
                type=MCPMessageType.ERROR,
                error=f"资源不存在: {uri}",
                id=message.id
            )
        
        resource = self.resources[uri]
        return MCPMessage(
            type=MCPMessageType.RESPONSE,
            result={
                "contents": [{
                    "uri": uri,
                    "mimeType": resource.mime_type,
                    "text": resource.content if isinstance(resource.content, str) else json.dumps(resource.content)
                }]
            },
            id=message.id
        )


print("=== MCPServer 实现 ===")
print("支持方法: tools/list, tools/call, resources/list, resources/read, ping")

## 14.3 创建示例 MCP 服务器

现在让我们创建一个实际的 MCP 服务器示例，包含天气查询工具和一些资源。

In [ ]:
def get_weather(city: str, units: str = "celsius") -> str:
    weather_data = {
        "北京": {"temp": 22, "condition": "晴", "humidity": 45},
        "上海": {"temp": 26, "condition": "多云", "humidity": 60},
        "广州": {"temp": 28, "condition": "阵雨", "humidity": 75},
        "深圳": {"temp": 27, "condition": "晴间多云", "humidity": 70}
    }
    
    data = weather_data.get(city, {"temp": 20, "condition": "未知", "humidity": 50})
    temp = data["temp"]
    if units == "fahrenheit":
        temp = temp * 9 / 5 + 32
    
    return f"{city}天气: {data['condition']}, 温度{temp}°{'C' if units == 'celsius' else 'F'}, 湿度{data['humidity']}%"


def calculate(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"


weather_server = MCPServer("WeatherServer")

weather_tool = MCPTool(
    name="get_weather",
    description="获取指定城市的天气信息",
    input_schema={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "城市名称"},
            "units": {"type": "string", "description": "温度单位 (celsius/fahrenheit)", "default": "celsius"}
        },
        "required": ["city"]
    },
    handler=get_weather
)

calc_tool = MCPTool(
    name="calculate",
    description="执行数学计算",
    input_schema={
        "type": "object",
        "properties": {
            "expression": {"type": "string", "description": "数学表达式"}
        },
        "required": ["expression"]
    },
    handler=calculate
)

weather_server.register_tool(weather_tool)
weather_server.register_tool(calc_tool)

city_info_resource = MCPResource(
    uri="resource://cities/info",
    name="城市信息",
    description="主要城市的基本信息",
    mime_type="application/json",
    content={
        "北京": {"population": "2154万", "province": "北京市"},
        "上海": {"population": "2428万", "province": "上海市"},
        "广州": {"population": "1867万", "province": "广东省"},
        "深圳": {"population": "1756万", "province": "广东省"}
    }
)

weather_server.register_resource(city_info_resource)

print("\n示例 MCP 服务器创建完成!")

## 14.4 MCP 客户端实现

MCP 客户端用于与服务器通信，发送请求并处理响应。下面实现一个简化的 MCP 客户端。

In [ ]:
class MCPClient:
    def __init__(self, server: MCPServer):
        self.server = server
        self.request_id = 0
        self.available_tools: List[Dict] = []
        self.available_resources: List[Dict] = []

    def _send_request(self, method: str, params: Dict = None) -> MCPMessage:
        self.request_id += 1
        request = MCPMessage(
            type=MCPMessageType.REQUEST,
            method=method,
            params=params or {},
            id=self.request_id
        )
        return self.server.handle_request(request)

    def initialize(self):
        print(f"[Client] 初始化连接到 {self.server.name}")
        
        ping_response = self._send_request("ping")
        print(f"[Client] Ping 响应: {ping_response.result}")
        
        tools_response = self._send_request("tools/list")
        self.available_tools = tools_response.result.get("tools", [])
        print(f"[Client] 可用工具: {[t['name'] for t in self.available_tools]}")
        
        resources_response = self._send_request("resources/list")
        self.available_resources = resources_response.result.get("resources", [])
        print(f"[Client] 可用资源: {[r['uri'] for r in self.available_resources]}")

    def call_tool(self, tool_name: str, **arguments) -> str:
        print(f"[Client] 调用工具: {tool_name} 参数={arguments}")
        response = self._send_request(
            "tools/call",
            {"name": tool_name, "arguments": arguments}
        )
        
        if response.type == MCPMessageType.ERROR:
            return f"错误: {response.error}"
        
        content = response.result.get("content", [])
        if content:
            return content[0].get("text", "")
        return ""

    def read_resource(self, uri: str) -> str:
        print(f"[Client] 读取资源: {uri}")
        response = self._send_request("resources/read", {"uri": uri})
        
        if response.type == MCPMessageType.ERROR:
            return f"错误: {response.error}"
        
        contents = response.result.get("contents", [])
        if contents:
            return contents[0].get("text", "")
        return ""


print("=== MCPClient 实现 ===")
client = MCPClient(weather_server)
client.initialize()

## 14.5 使用 MCP 客户端

现在让我们使用客户端调用服务器的工具和资源。

In [ ]:
print("\n=== 工具调用示例 ===")

result1 = client.call_tool("get_weather", city="北京")
print(f"结果: {result1}")

result2 = client.call_tool("get_weather", city="上海", units="fahrenheit")
print(f"结果: {result2}")

result3 = client.call_tool("calculate", expression="25 * 4 + 10")
print(f"结果: {result3}")

print("\n=== 资源读取示例 ===")

resource_content = client.read_resource("resource://cities/info")
print(f"资源内容:\n{resource_content}")

## 14.6 LangChain 与 MCP 集成

MCP 可以与 LangChain 集成，让 LangChain 的 Agent 能够使用 MCP 服务器提供的工具。下面模拟这种集成方式。

In [ ]:
class MCPToolAdapter:
    def __init__(self, mcp_tool: MCPTool, client: MCPClient):
        self.mcp_tool = mcp_tool
        self.client = client
        self.name = mcp_tool.name
        self.description = mcp_tool.description

    def run(self, **kwargs) -> str:
        return self.client.call_tool(self.name, **kwargs)


class LangChainMCPAgent:
    def __init__(self, client: MCPClient):
        self.client = client
        self.tools = [
            MCPToolAdapter(weather_server.tools[t_name], client)
            for t_name in weather_server.tools
        ]

    def think(self, query: str) -> str:
        print(f"\n[Agent] 用户查询: {query}")
        
        if "天气" in query or "温度" in query:
            cities = ["北京", "上海", "广州", "深圳"]
            for city in cities:
                if city in query:
                    result = self.client.call_tool("get_weather", city=city)
                    return f"[Agent] 使用工具查询到: {result}"
        
        if "计算" in query or "算" in query:
            import re
            match = re.search(r'[\d+\-*/().\s]+', query)
            if match:
                expr = match.group().strip()
                result = self.client.call_tool("calculate", expression=expr)
                return f"[Agent] 使用工具计算: {result}"
        
        return f"[Agent] 我可以帮你查询天气或进行数学计算，请告诉我具体需求。"


print("=== LangChain 与 MCP 集成 ===")

agent = LangChainMCPAgent(client)

queries = [
    "北京今天天气怎么样？",
    "帮我计算 125 + 64 * 3",
    "上海的温度是多少？"
]

for q in queries:
    response = agent.think(q)
    print(response)

## 14.7 MCP 资源订阅

MCP 协议还支持资源订阅，当资源变化时服务器可以主动通知客户端。下面实现这一功能。

In [ ]:
class MCPServerWithSubscription(MCPServer):
    def __init__(self, name: str):
        super().__init__(name)
        self.subscriptions: Dict[str, List[Callable]] = {}

    def subscribe(self, uri: str, callback: Callable):
        if uri not in self.subscriptions:
            self.subscriptions[uri] = []
        self.subscriptions[uri].append(callback)
        print(f"[Server] 新增订阅: {uri}")

    def update_resource(self, uri: str, new_content: Any):
        if uri not in self.resources:
            print(f"[Server] 资源不存在: {uri}")
            return
        
        self.resources[uri].content = new_content
        print(f"[Server] 资源已更新: {uri}")
        
        if uri in self.subscriptions:
            for callback in self.subscriptions[uri]:
                try:
                    callback(uri, new_content)
                except Exception as e:
                    print(f"[Server] 通知失败: {e}")


class MCPClientWithSubscription(MCPClient):
    def __init__(self, server: MCPServerWithSubscription):
        super().__init__(server)
        self.server = server

    def subscribe_to_resource(self, uri: str):
        def on_update(updated_uri: str, content: Any):
            print(f"[Client] 收到资源更新通知: {updated_uri}")
            print(f"[Client] 新内容: {content}")
        
        self.server.subscribe(uri, on_update)
        print(f"[Client] 已订阅资源: {uri}")


print("=== MCP 资源订阅 ===")

sub_server = MCPServerWithSubscription("SubscriptionServer")
sub_server.register_resource(city_info_resource)

sub_client = MCPClientWithSubscription(sub_server)
sub_client.initialize()

sub_client.subscribe_to_resource("resource://cities/info")

print("\n[演示] 更新资源...")
updated_content = city_info_resource.content.copy()
updated_content["北京"]["population"] = "2160万"
sub_server.update_resource("resource://cities/info", updated_content)

## 练习

1. **扩展 MCP 服务器**：创建一个新的 MCP 服务器，提供文件系统操作工具（如读取文件、列出目录）和相应的资源。

2. **工具链调用**：实现一个多步骤工具调用流程，让 Agent 能够自动决定何时使用哪个 MCP 工具。

3. **资源版本控制**：扩展 MCP 资源系统，支持资源版本历史和回滚功能。

4. **多服务器集成**：创建一个客户端能够同时连接多个 MCP 服务器，统一管理来自不同服务器的工具和资源。